In [1]:
import pandas as pd

# The Palmer Archipelago Penguin Data

In [2]:
# load the data
url = 'https://raw.githubusercontent.com/um-perez-alvaro/Data-Science-Theory/master/Data/penguins_size.csv'
data = pd.read_csv(url)
data.head(5)

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


Data were collected and made available by Dr. Kristen Gorman and the Palmer Station, Antarctica LTER, a member of the Long Term Ecological Research Network.
This dataset contains data for 344 penguins.
There are 3 different species of penguins in this dataset, collected from 3 islands in the Palmer Archipelago, Antarctica.

In [3]:
data.species.value_counts()

Adelie       152
Gentoo       124
Chinstrap     68
Name: species, dtype: int64

The culmen is the upper ridge of a bird’s bill. For this penguin data, the culmen (bill) length and depth are measured as shown below.

The **goal** is to predict the penguin species from physical measurements (culmen length, culmen depth, flipper length, and body mass). 

**Part 1:** Import and instantiate a k-nearest neighbors model.

In [4]:
# your  code here
from sklearn.neighbors import KNeighborsClassifier

knn_clf = KNeighborsClassifier()

**Part 2:** Define the feature matrix X and the target vector y from the dataframe, and then split X and y into training and testing sets.

In [14]:
feature_cols = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']

In [16]:
# your  code here
y = data.species
X = data[feature_cols]

In [17]:
# split X and y into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.75) # default is 0.75

**Part 3:** Build a pipeline that consists of two steps: an imputer (to fill missing values with the mean value) and the knn classifier.

In [29]:
# your  code here

#Preprocessing 
from sklearn.impute import SimpleImputer

# Pipeline
from sklearn.pipeline import Pipeline   



In [19]:
pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('clf', knn_clf)
])

**Part 4:** Use a grid search to tune in the `n_neighbors` and `weights` hyperparameters

<div class="admonition note alert alert-info">
<p class="first admonition-title" style="font-weight: bold;">Note</p>
<p>  The Palmer Archipelago Penguin Data is relatively small.
    Don't use a large value for the <tt> cv </tt> parameter</p> (Javier would use <tt> cv=5</tt>.)
</div>

In [20]:
# Grid search 
from sklearn.model_selection import GridSearchCV

In [21]:
# your  code here
param_grid = { 
    'clf__n_neighbors': list(range(1,21)),
    'clf__weights' : ['uniform','distance']
}

In [22]:
# instantiate and fit the grid
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('imputer', SimpleImputer()),
                                       ('clf', KNeighborsClassifier())]),
             n_jobs=-1,
             param_grid={'clf__n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11,
                                              12, 13, 14, 15, 16, 17, 18, 19,
                                              20],
                         'clf__weights': ['uniform', 'distance']},
             scoring='accuracy', verbose=1)

In [23]:
# view the results
pd.DataFrame(grid.cv_results_)[['mean_test_score', 'params']]

,mean_test_score,params
0,0.802187,"{'clf__n_neighbors': 1, 'clf__weights': 'unifo..."
1,0.802187,"{'clf__n_neighbors': 1, 'clf__weights': 'dista..."
2,0.767270,"{'clf__n_neighbors': 2, 'clf__weights': 'unifo..."
3,0.802187,"{'clf__n_neighbors': 2, 'clf__weights': 'dista..."
4,0.782655,"{'clf__n_neighbors': 3, 'clf__weights': 'unifo..."
5,0.790498,"{'clf__n_neighbors': 3, 'clf__weights': 'dista..."
6,0.767195,"{'clf__n_neighbors': 4, 'clf__weights': 'unifo..."
7,0.802187,"{'clf__n_neighbors': 4, 'clf__weights': 'dista..."
8,0.783107,"{'clf__n_neighbors': 5, 'clf__weights': 'unifo..."
9,0.794419,"{'clf__n_neighbors': 5, 'clf__weights': 'dista..."


In [24]:


# best hyper-parameters
grid.best_params_



{'clf__n_neighbors': 1, 'clf__weights': 'uniform'}

In [25]:


# best predictor
best_clf = grid.best_estimator_



**Part 5:** Use accuracy and the confusion matrix to evaluate the performance of your best model on the test set

In [26]:
# your  code here


from sklearn.metrics import confusion_matrix, accuracy_score

y_test_pred = best_clf.predict(X_test)

In [27]:
# accuracy
accuracy_score(y_test,y_test_pred)

0.8023255813953488

In [28]:
# confusion matrix
confusion_matrix(y_test,y_test_pred)

array([[28,  4,  4],
       [ 6,  7,  0],
       [ 3,  0, 34]])

<div class="alert alert-block alert-danger"> <tt>KNeighborsClassifier </tt> is a distance based model. Distance based models are affected by the scale of the features. They give higher weightage to features which have higher magnitude (The <tt>body_mass_g</tt> feature in our case).
We do not want our classifier to be affected by the magnitude of the features. To overcome this problem, we can bring down all the variables to the same scale. </div>

**Part 6:** Add a `StandardScaler` to your pipeline, and repeat steps 4 and 5.

In [33]:
# your  code here

from sklearn.preprocessing import StandardScaler

pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('kclf', knn_clf)
])

In [34]:
param_grid = { 
    'kclf__n_neighbors': list(range(1,21)),
    'kclf__weights' : ['uniform','distance']
}

In [35]:
# instantiate and fit the grid
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('imputer', SimpleImputer()),
                                       ('scaler', StandardScaler()),
                                       ('kclf', KNeighborsClassifier())]),
             n_jobs=-1,
             param_grid={'kclf__n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
                                               11, 12, 13, 14, 15, 16, 17, 18,
                                               19, 20],
                         'kclf__weights': ['uniform', 'distance']},
             scoring='accuracy')

In [36]:
# best predictor
best_clf = grid.best_estimator_

In [37]:
# best hyper-parameters
grid.best_params_

{'kclf__n_neighbors': 3, 'kclf__weights': 'uniform'}

In [38]:
y_test_pred = best_clf.predict(X_test)

In [39]:
# accuracy
accuracy_score(y_test,y_test_pred)

0.9767441860465116

In [41]:
# much improved

In [40]:
# confusion matrix
confusion_matrix(y_test,y_test_pred)

array([[35,  1,  0],
       [ 0, 13,  0],
       [ 1,  0, 36]])